## Домашнее задание №4 (курс "Python 1")

### Выполнил: <font color='red'>Фоменко Елизавета Антоновна</font>

### Тема: Web-сервер для обучения и использования ML-моделей

#### Преподаватели: Роман Ищенко (roman.ischenko@gmail.com) и Илья Склонин

**Дедлайн**: 18.01.2026

**Среда выполнения**: Jupyter Notebook (Python 3.9+)

#### Правила:

Результаты выполнения задания:

- архив со скриптами и файлами Dockerfile, который 1-2 команды позволяет развернуть сервер, решающий поставленные в задании задачи
- Jupyter Notebook, где __весь код__ из скриптов дублируется (1 ячейка - 1 скрипт) с комментарием, содержащим информацию о том, из какого файла взят код и что верхнеуровнево этот код делает

__Максимальное число баллов за задание - 35__.

Готовое задание отправляется на почту преподавателя.

Задание выполняется самостоятельно. Если какие-то студенты будут уличены в списывании, все они автоматически получат за эту работу 0 баллов. Если вы нашли в Интернете какой-то специфичный код, который собираетесь заимствовать, обязательно укажите это в задании - наверняка вы не единственный, кто найдёт и использует эту информацию.

Удалять фрагменты формулировок заданий запрещается.

### Постановка задачи:

**Серверная часть (22 балла):**

- В данной работе нужно написать многозадачный веб-сервер для обучения и инференса ML моделей. На старте сервер получает на вход (через .env) конфиг, в котором должны быть указаны 3 параметра: путь к директории для сохранения моделей внутри контейнера сервера, число ядер, доступных для обучения и максимальное число моделей, которые могут быть одновременно загружены для инференса.


- Сервер должен реализовывать следующие методы:
    - `fit(X, y, config)` - обучить модель и сохранить на диск по указанным именем
    - `predict(y, config)` - предсказать с помощью обученной и загруженной модели по её имени
    - `load(config)` - загрузить обученную модель по её имени в режим инференса
    - `unload(config)` - выгрузить загруженную модель по её имени
    - `remove(config)` - удалить обученную модель с диска по её имени
    - `remove_all()` - удалить все обученные модели с диска


- Содержимое конфигов и форматы данных предлагается продумать и реализовать самостоятельно
- Сервер должен иметь счётчик активных процессов. Максимальное число активных процессов соответствует числу ядер, переданному в конфиге при старте сервиса. Каждое обучение модели запускается в отдельном процессе и до своего завершения потребляет этот процесс. Один процесс всегда остаётся для сервера, в нём же загружаются и работают на инференс обученные модели
- Сервер должен корректно обрабатывать все граничные случаи (запуск обучения без свободных ядер, запуск инфренса свыше лимита, запросы с несуществующими именами моделей, запросы с дублирующимися именами моделей)
- В реализации должны поддерживаться не менее трёх дискриминативных моделей (т.е. принимающих на вход объекты и метки при обучении и предсказывающих метки для новых объектов)
- Сервер должен быть реализован на FastAPI
- Проект разворачивается с помощью выбранной библиотеки управления виртуальными окружениями и технологии контейнеризации Docker

**Клиентская часть (13 баллов):**

- Клиентская часть должна демонстрировать работу с реализованным сервером с помощью библиотек requests и aiohttp. Она может быть реализована непосредственно в Jupyter Notebook, с описанием ожидаемого действия, или в отдельном(-ых) скрипте(-ах), с дублированием в Jupyter Notebook (тогда работоспособность в ноутбуке не требуется). Далее описываются отдельные функции:
- Код вызова последовательного вызова обучения как минимум двух (N) различных моделей с таким набором данных и параметрами, чтобы обучение одной модели длилось не менее 60 секунд.
- Код вызова асинхронного вызова обучения как минимум двух различных моделей с демонстрацией, что работа выполняется в два (в N) раза быстрее
- Асинхронный вызов нескольких предсказаний
- Код демонстрации остальных функций сервера (загрузка, выгрузка, удаление)
- Должны обрабатываться ошибки и исключения, возвращаемые сервером


## Серверна часть

**пример файла .env**

```
dir_path=./saved_models
core_nums=4
max_models=3
```

In [ ]:
# process_env.py
# файл для обработки .env файла, в котором находятся настройки сервера


# пример, как юзать dotenv для обработки .env файлов
# https://www.geeksforgeeks.org/python/read-environment-variables-with-python-dotenv/

import os
from dotenv import load_dotenv

load_dotenv()

# загружаем переменные из env и ставим параметры по умолчанию
dir_path = os.getenv("dir_path", "./saved_models")               # путь к директории для сохранения моделей внутри контейнера сервера
core_nums = int(os.getenv("core_nums", 5))                       # число ядер, доступных для обучения
max_models = int(os.getenv("max_models", 3))                     # максимальное число моделей, которые могут быть одновременно загружены для инференса

# создаем директорию для загруженных моделей
os.makedirs(dir_path, exist_ok=True)

In [ ]:
# models.py
# файл, в котором описываются все функции для работы с моделями (обучение, предсказание, загрузка/удаление и так далее)


# для сохранения моделей юзаем pickle, как советует WANDB: https://wandb.ai/a-sh0ts/publications/reports/How-to-Save-a-Classifier-to-Disk-in-Scikit-learn--Vmlldzo0NDc1ODI0

import os
import pickle
from .process_env import *
from sklearn.linear_model import LogisticRegression, LinearRegression, SGDClassifier, SGDRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
import numpy as np


# Словарь для загруженных моделей

loaded_models = {}

MODELS_REGISTRY = {
    # classificators
    'logistic_clf': LogisticRegression,
    'sgd_clf': SGDClassifier,
    'tree_clf': DecisionTreeClassifier,
    'forest_clf': RandomForestClassifier,
    'gb_clf': GradientBoostingClassifier,

    # regressions
    'linear_reg': LinearRegression,
    'sgd_reg': SGDRegressor,
    'tree_reg': DecisionTreeRegressor,
    'forest_reg': RandomForestRegressor,
    'gb_reg': GradientBoostingRegressor,
}


def fit_model(X, y, config):
    cfg = config.copy()

    name = cfg.pop('name')
    model_path = os.path.join(dir_path, f'{name}.pkl')

    X = np.array(X)
    y = np.array(y)

    # если имена моделей дублируются, кидаем error и сообщаем о повторке
    if os.path.exists(model_path):
        raise ValueError(f'Unable to create model with name {name}: file already exists')

    # модель = класс из sklearn с fit, fit_tranform, predict методами
    model = cfg.pop('model')
    if model not in MODELS_REGISTRY.keys():
        raise ValueError(f'Unsupported model type')
    model = MODELS_REGISTRY[model](**cfg)

    model.fit(X, y)

    # сохраняем модель
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    return {'status': 'success', 'message': f'Model {model} with name {name} is fit and saved to {model_path}'}


def predict_model(X, config):
    cfg = config.copy()
    name = cfg.pop('name')

    X = np.array(X)

    if name not in loaded_models:
        raise ValueError(f'Model with name {name} is not loaded')

    model = loaded_models[name]
    y_pred = model.predict(X)

    return y_pred.tolist()


def load_model(config):

    if len(loaded_models.keys()) >= max_models:
        raise ValueError(f'Number of loaded models is already the maximum possible')

    name = config['name']

    # если уже загружено, то ниче не делаем, но сообщаем об этом
    if name in loaded_models:
        return {'status': 'warning', 'message': f'Model {name} is already loaded'}

    model_path = os.path.join(dir_path, f'{name}.pkl')

    # запросы с несуществующими именами моделей
    if not os.path.exists(model_path):
        raise ValueError(f'Loading the model {name} from {model_path} failed: no such file')

    # загружаем модель и добавляем ее в loaded_models
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    loaded_models[name] = model
    return {'status': 'success', 'message': f'Model {name} is loaded'}


def unload_model(config):
    name = config['name']

    if name not in loaded_models.keys():
        raise ValueError(f'Model with name {name} is not loaded')

    del loaded_models[name]
    return {'status': 'success', 'message': f'Model {name} is unloaded'}


def remove_model(config):
    name = config['name']
    model_path = os.path.join(dir_path, f'{name}.pkl')

    if not os.path.exists(model_path):
        raise ValueError(f'No such file {model_path}')

    os.remove(model_path)

    if name in loaded_models:
        del loaded_models[name]

    return {'status': 'success', 'message': f'Model {name} removed'}


def remove_all_models():
    for file in os.listdir(dir_path):
        if file.endswith(".pkl"):
            os.remove(os.path.join(dir_path, file))
    loaded_models.clear()
    return {'status': 'success', 'message': f'All models removed'}

In [ ]:
# main.py
# файл с сервером fastAPI

# app/main.py
from fastapi import FastAPI, HTTPException
from concurrent.futures import ProcessPoolExecutor
from pydantic import BaseModel
import numpy as np
import asyncio
from .models import *
from .process_env import *
from multiprocessing import Value, Lock

import time


app = FastAPI(title="ML MODELS INFERENCE SERVER")

cores_for_fit = core_nums - 1
fit_pool = ProcessPoolExecutor(max_workers=cores_for_fit)


# https://blog.jetbrains.com/de/pycharm/2025/08/schnelleres-python-entfernen-des-global-interpreter-lock-in-python/
active_proc_number = Value("i", 0)
lock = Lock()


def fit_job(X, y, config):
        X = np.asarray(X)
        y = np.asarray(y)
        return fit_model(X, y, config)



# как красиво валидировать реквесты: буквально с главной страницы FastAPI: https://fastapi.tiangolo.com/#recap
class FitRequest(BaseModel):
    X: list
    y: list
    config: dict

class PredictRequest(BaseModel):
    X: list
    config: dict

class BasicRequest(BaseModel):
    config: dict


@app.get('/')
def root():
    return {'message': 'Server for models inference is running!', 'models will be saved to': dir_path,
            'maximum fit processes': cores_for_fit, 'maximum models to load': max_models}

@app.get("/status")
def status():
    with lock:
        active_processees = active_proc_number.value

    return {'maximum fit processes': cores_for_fit, 'active processes': active_processees, 'empty processes': cores_for_fit - active_processees}

# https://medium.com/@AlexanderObregon/understanding-pythons-multiprocessing-module-744dba8d4be4
@app.post("/fit")
async def fit_endpoint(request: FitRequest):
    loop = asyncio.get_running_loop()

    with lock:
        active_proc_number.value += 1
    try:
        result = await loop.run_in_executor(
            fit_pool,
            fit_job,
            request.X,
            request.y,
            request.config,
        )
        return result

    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))

    finally:
        with lock:
            active_proc_number.value -= 1
МИ


@app.post("/predict")
def predict_endpoint(request: PredictRequest):
    try:
        result = predict_model(request.X, request.config)
        return result

    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/load")
def load_endpoint(request: BasicRequest):
    try:
        result = load_model(request.config)
        return result

    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))


@app.post("/unload")
def unload_endpoint(request: BasicRequest):
    try:
        result = unload_model(request.config)
        return result

    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/remove")
def remove_endpoint(request: BasicRequest):
    try:
        result = remove_model(request.config)
        return result

    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/remove_all")
def remove_all_endpoint():
    try:
        result = remove_all_models()
        return result

    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))

## Клиентская часть

In [ ]:
## test_request.py

# test.py
import requests
import time
import json

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

from scipy.stats import pearsonr


X, y = make_regression(n_samples = 1500000,     #1500000
                        n_features = 50,
                        random_state = 57,
                        noise = 0.5)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 57, test_size = 0.2)


url = "http://127.0.0.1:8000"


# стартовая информация про сервер
resp = requests.get(f"{url}/")
print('ROOT SERVER INFO')
print(json.dumps(resp.json(), indent = 5), '\n')



# статус сервера
resp = requests.get(f"{url}/status")
print('SERVER STATUS INFO')
print(json.dumps(resp.json(), indent = 5), '\n')



# here we assign configa information for our future models

linreg_config = {'name': 'LinearRegression',
                'model': 'linear_reg',
                'fit_intercept': True}

sgdreg_config = {'name': 'SGDRegressor',
                'model': 'sgd_reg',
                'tol': 1e-6,
                'random_state': 57,
                'penalty': 'l2',
                'alpha': 0.05}

dectree_config = {'name': 'DecisionTreeRegressor',
                'model': 'tree_reg',
                'max_depth': 10,
                'min_samples_leaf': 2,
                'random_state': 57}

rforest_config = {'name': 'RandomForestRegressor',
                'model': 'forest_reg',
                'max_depth': 5,
                'n_estimators':20,
                'random_state': 57}

gboost_config = {'name': 'GradientBoostingRegressor',
                'model': 'gb_reg',
                'learning_rate': 0.05,
                'n_estimators': 20,
                'max_depth': 5,
                'random_state': 57}


configs = [linreg_config, sgdreg_config, dectree_config, rforest_config, gboost_config]


# fit models using REQUEST

print('-'*40, 'FIT MODELS', '-'*40, '\n')
total_time = 0

for config in configs:
    model = {'X': X_train.tolist(), 'y': y_train.tolist(), 'config': config}
    start_time = time.time()
    resp = requests.post(f"{url}/fit", json=model)
    end_time = time.time()
    if resp.status_code == 200:
        print(json.dumps(resp.json(), indent = 5), '\n')
        print(f'time to fit the model: {end_time - start_time:.3f}', '\n')
        total_time += end_time - start_time
    else:
        print(f'error: {resp.text}')

    resp = requests.get(f"{url}/status")
    print('SERVER STATUS')
    print(json.dumps(resp.json(), indent = 5), '\n')

print('-'*(len('TOTAL TIME for models fitting with request: ') + len(str(total_time)) + 4))
print(f'| TOTAL TIME for models fitting with request: {total_time} |')
print('-'*(len('TOTAL TIME for models fitting with request: ') + len(str(total_time)) + 4), '\n\n')


# predictions
print('-'*40, 'MAKE PREDICTIONS', '-'*40, '\n')

for i in range(len(configs)):
    config = configs[i]
    resp = requests.post(f'{url}/load', json={'config': config})
    if resp.status_code != 200:
        print(f'error: {resp.text}')                  # значит уже все занято, надо что-то удалить
        resp = requests.post(f'{url}/unload', json={'config': configs[i-1]})
        print(json.dumps(resp.json(), indent = 5), '\n')
        resp = requests.post(f'{url}/load', json={'config': config})

    print(json.dumps(resp.json(), indent = 5), '\n')
    predict_data = {'X': X_test.tolist(), 'config': config}
    resp = requests.post(f'{url}/predict', json = predict_data)
    y_pred = resp.json()
    print(f'Pearson correlation for {config["name"]}: {pearsonr(y_test, y_pred).statistic:.4f}', '\n')

print('\n')
resp = requests.post(f'{url}/remove', json={'config': configs[0]})
if resp.status_code == 200:
    print(json.dumps(resp.json(), indent = 5), '\n')
else:
    print(f'error: {resp.text}')

print('removing all the models.....')

resp = requests.post(f"{url}/remove_all")
print(json.dumps(resp.json(), indent = 5), '\n')



In [ ]:
# тут использован весь функционал сервера, к тому же при предикшнс каждая модель загружается, идет проверка
# на превышение доступного количества загруженных моделей, если из сервера поступает ошибка, то
# предыдущая модель unloads, и снова скачивается нужная модель

**вывод при запуске кода**

```
ROOT SERVER INFO
{
     "message": "Server for models inference is running!",
     "models will be saved to": "./saved_models",
     "maximum fit processes": 3,
     "maximum models to load": 3
}

SERVER STATUS INFO
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

---------------------------------------- FIT MODELS ----------------------------------------

{
     "status": "success",
     "message": "Model LinearRegression() with name LinearRegression is fit and saved to ./saved_models/LinearRegression.pkl"
}

time to fit the model: 85.886

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

{
     "status": "success",
     "message": "Model SGDRegressor(alpha=0.05, random_state=57, tol=1e-06) with name SGDRegressor is fit and saved to ./saved_models/SGDRegressor.pkl"
}

time to fit the model: 74.892

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

{
     "status": "success",
     "message": "Model DecisionTreeRegressor(max_depth=10, min_samples_leaf=2, random_state=57) with name DecisionTreeRegressor is fit and saved to ./saved_models/DecisionTreeRegressor.pkl"
}

time to fit the model: 131.082

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

{
     "status": "success",
     "message": "Model RandomForestRegressor(max_depth=5, n_estimators=20, random_state=57) with name RandomForestRegressor is fit and saved to ./saved_models/RandomForestRegressor.pkl"
}

time to fit the model: 583.665

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

{
     "status": "success",
     "message": "Model GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=20,\n                          random_state=57) with name GradientBoostingRegressor is fit and saved to ./saved_models/GradientBoostingRegressor.pkl"
}

time to fit the model: 918.203

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

------------------------------------------------------------------
| TOTAL TIME for models fitting with request: 1793.7273070812225 |
------------------------------------------------------------------


---------------------------------------- MAKE PREDICTIONS ----------------------------------------

{
     "status": "success",
     "message": "Model LinearRegression is loaded"
}

Pearson correlation for LinearRegression: 1.0000

{
     "status": "success",
     "message": "Model SGDRegressor is loaded"
}

Pearson correlation for SGDRegressor: 1.0000

{
     "status": "success",
     "message": "Model DecisionTreeRegressor is loaded"
}

Pearson correlation for DecisionTreeRegressor: 0.8864

error: {"detail":"Number of loaded models is already the maximum possible"}
{
     "status": "success",
     "message": "Model DecisionTreeRegressor is unloaded"
}

{
     "status": "success",
     "message": "Model RandomForestRegressor is loaded"
}

Pearson correlation for RandomForestRegressor: 0.7630

error: {"detail":"Number of loaded models is already the maximum possible"}
{
     "status": "success",
     "message": "Model RandomForestRegressor is unloaded"
}

{
     "status": "success",
     "message": "Model GradientBoostingRegressor is loaded"
}

Pearson correlation for GradientBoostingRegressor: 0.8929



{
     "status": "success",
     "message": "Model LinearRegression removed"
}

removing all the models.....
{
     "status": "success",
     "message": "All models removed"
}
```

In [ ]:
# test_aiohttp.py


import asyncio
import aiohttp
import time
import json
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

X, y = make_regression(n_samples = 1500000,
                        n_features = 50,
                        random_state = 57,
                        noise = 0.5)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 57, test_size = 0.2)


url = "http://127.0.0.1:8000"

# here we assign configa information for our future models

linreg_config = {'name': 'LinearRegression',
                'model': 'linear_reg',
                'fit_intercept': True}

sgdreg_config = {'name': 'SGDRegressor',
                'model': 'sgd_reg',
                'tol': 1e-6,
                'random_state': 57,
                'penalty': 'l2',
                'alpha': 0.05}

dectree_config = {'name': 'DecisionTreeRegressor',
                'model': 'tree_reg',
                'max_depth': 10,
                'min_samples_leaf': 2,
                'random_state': 57}

rforest_config = {'name': 'RandomForestRegressor',
                'model': 'forest_reg',
                'max_depth': 5,
                'n_estimators':20,
                'random_state': 57}

gboost_config = {'name': 'GradientBoostingRegressor',
                'model': 'gb_reg',
                'learning_rate': 0.05,
                'n_estimators': 20,
                'max_depth': 5,
                'random_state': 57}


configs = [linreg_config, sgdreg_config, dectree_config, rforest_config, gboost_config]


# # https://docs.aiohttp.org/en/stable/client_quickstart.html


async def fetch(session, method, endpoint, json_data=None):
    async with session.request(method, f"{url}/{endpoint}", json=json_data) as resp:
        try:
            return resp.status, await resp.json()
        except:
            return resp.status, await resp.text()

async def fit_model(session, config, X_train, y_train):
    model = {'X': X_train.tolist(), 'y': y_train.tolist(), 'config': config}
    start_time = time.time()
    status_code, result = await fetch(session, 'POST', 'fit', model)
    end_time = time.time()
    if status_code == 200:
        print(json.dumps(result, indent=5), '\n')
        print(f'time to fit the model: {end_time - start_time:.3f}', '\n')
    else:
        print(f'error: {result}')
    status_s, result_s = await fetch(session, 'GET', 'status')
    print('SERVER STATUS')
    print(json.dumps(result_s, indent=5), '\n')
    return end_time - start_time


async def get_predictions(session, config, X_test, y_test, prev_config=None):
    status_code, result = await fetch(session, 'POST', 'load', {'config': config})
    if status_code != 200:
        await fetch(session, 'POST', 'unload', {'config': prev_config})
        status_code, result = await fetch(session, 'POST', 'load', {'config': config})

    print(json.dumps(result, indent=5), '\n')

    status_code, y_pred = await fetch(session, 'POST', 'predict', {'X': X_test.tolist(), 'config': config})
    print(f'Pearson correlation for {config["name"]}: {pearsonr(y_test, y_pred).statistic:.4f}', '\n')


timeout = aiohttp.ClientTimeout(total=10000)
async def pipeline():
    async with aiohttp.ClientSession(timeout = timeout) as session:
        status_code, result = await fetch(session, 'GET', '')
        print('ROOT SERVER INFO')
        print(json.dumps(result, indent=5), '\n')

        status_code, result = await fetch(session, 'GET', 'status')
        print('SERVER STATUS INFO')
        print(json.dumps(result, indent=5), '\n')

        print('-'*40, 'FIT MODELS', '-'*40, '\n')
        start_time = time.time()
        tasks = [fit_model(session, config, X_train, y_train) for config in configs]
        await asyncio.gather(*tasks)
        end_time = time.time()
        total_time = end_time - start_time

        print('-'*(len('TOTAL TIME for models fitting with request: ') + len(str(total_time)) + 4))
        print(f'| TOTAL TIME for models fitting with request: {total_time} |')
        print('-'*(len('TOTAL TIME for models fitting with request: ') + len(str(total_time)) + 4), '\n\n')


        print('-'*40, 'MAKE PREDICTIONS', '-'*40, '\n')
        prev_config = None
        for config in configs:
            await get_predictions(session, config, X_test, y_test, prev_config)
            prev_config = config

        status, result = await fetch(session, 'POST', 'remove', {'config': configs[0]})
        if status == 200:
            print(json.dumps(result, indent=5), '\n')
        else:
            print(f'error: {result}')

        print('removing all the models.....')
        status, result = await fetch(session, 'POST', 'remove_all')
        print(json.dumps(result, indent=5), '\n')

asyncio.run(pipeline())

```
ROOT SERVER INFO
{
     "message": "Server for models inference is running!",
     "models will be saved to": "./saved_models",
     "maximum fit processes": 3,
     "maximum models to load": 3
}

SERVER STATUS INFO
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

---------------------------------------- FIT MODELS ----------------------------------------

{
     "status": "success",
     "message": "Model LinearRegression() with name LinearRegression is fit and saved to ./saved_models/LinearRegression.pkl"
}

time to fit the model: 83.176

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 3,
     "empty processes": 0
}

{
     "status": "success",
     "message": "Model SGDRegressor(alpha=0.05, random_state=57, tol=1e-06) with name SGDRegressor is fit and saved to ./saved_models/SGDRegressor.pkl"
}

time to fit the model: 71.026

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 3,
     "empty processes": 0
}

{
     "status": "success",
     "message": "Model DecisionTreeRegressor(max_depth=10, min_samples_leaf=2, random_state=57) with name DecisionTreeRegressor is fit and saved to ./saved_models/DecisionTreeRegressor.pkl"
}

time to fit the model: 138.827

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 2,
     "empty processes": 1
}

{
     "status": "success",
     "message": "Model RandomForestRegressor(max_depth=5, n_estimators=20, random_state=57) with name RandomForestRegressor is fit and saved to ./saved_models/RandomForestRegressor.pkl"
}

time to fit the model: 568.003

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 1,
     "empty processes": 2
}

{
     "status": "success",
     "message": "Model GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=20,\n                          random_state=57) with name GradientBoostingRegressor is fit and saved to ./saved_models/GradientBoostingRegressor.pkl"
}

time to fit the model: 902.905

SERVER STATUS
{
     "maximum fit processes": 3,
     "active processes": 0,
     "empty processes": 3
}

-----------------------------------------------------------------
| TOTAL TIME for models fitting with request: 984.8720147916397 |
-----------------------------------------------------------------


---------------------------------------- MAKE PREDICTIONS ----------------------------------------

{
     "status": "success",
     "message": "Model LinearRegression is loaded"
}

Pearson correlation for LinearRegression: 1.0000

{
     "status": "success",
     "message": "Model SGDRegressor is loaded"
}

Pearson correlation for SGDRegressor: 1.0000

{
     "status": "success",
     "message": "Model DecisionTreeRegressor is loaded"
}

Pearson correlation for DecisionTreeRegressor: 0.8584

{
     "status": "success",
     "message": "Model RandomForestRegressor is loaded"
}

Pearson correlation for RandomForestRegressor: 0.8533

{
     "status": "success",
     "message": "Model GradientBoostingRegressor is loaded"
}

Pearson correlation for GradientBoostingRegressor: 0.9010

{
     "status": "success",
     "message": "Model LinearRegression removed"
}

removing all the models.....
{
     "status": "success",
     "message": "All models removed"
}
```